# Building a Chatbot and QA System
## Learning Objectives
- Build a retrieval-based QA system
- Implement a simple chatbot with rule-based and ML approaches
- Use LangChain for document Q&A


In [ ]:
# Rule-based chatbot
class RuleBasedChatbot:
    def __init__(self):
        self.rules = {
            ('hello', 'hi', 'hey'): 'Hello! How can I help you with data science today?',
            ('what is machine learning',): 'Machine learning is a subset of AI that enables systems to learn from data.',
            ('what is deep learning',): 'Deep learning uses neural networks with many layers to learn complex patterns.',
            ('thank', 'thanks'): 'You are welcome! Keep learning!',
            ('bye', 'goodbye'): 'Goodbye! Happy coding!'
        }

    def respond(self, user_input):
        user_lower = user_input.lower()
        for keywords, response in self.rules.items():
            if any(kw in user_lower for kw in keywords):
                return response
        return 'I am not sure about that. Can you rephrase your question?'

bot = RuleBasedChatbot()
questions = ['Hello!', 'What is machine learning?', 'Thank you!', 'Goodbye!']
for q in questions:
    print(f'User: {q}')
    print(f'Bot:  {bot.respond(q)}\n')

In [ ]:
# TF-IDF based retrieval QA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

knowledge_base = [
    'Python is a high-level programming language used widely in data science.',
    'Pandas is a data manipulation library built on NumPy.',
    'NumPy provides support for large multi-dimensional arrays and matrices.',
    'Scikit-learn offers simple tools for machine learning in Python.',
    'TensorFlow is an open-source deep learning framework by Google.',
    'Deep learning models require large amounts of labeled training data.'
]

vectorizer = TfidfVectorizer()
kb_vectors = vectorizer.fit_transform(knowledge_base)

def answer_question(question, top_k=2):
    q_vec = vectorizer.transform([question])
    sims = cosine_similarity(q_vec, kb_vectors).flatten()
    top_idx = sims.argsort()[-top_k:][::-1]
    for idx in top_idx:
        if sims[idx] > 0.1:
            print(f'  [{sims[idx]:.3f}] {knowledge_base[idx]}')

print('Q: What is pandas?')
answer_question('What is pandas?')
print('\nQ: How does deep learning work?')
answer_question('How does deep learning work?')

In [ ]:
# LangChain RAG pattern (conceptual)
langchain_example = '''
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.llms import HuggingFacePipeline

# 1. Load and split documents
loader = TextLoader("my_docs.txt")
docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)

# 2. Create vector store
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)

# 3. Build QA chain
qa_chain = RetrievalQA.from_chain_type(llm=llm, retriever=vectorstore.as_retriever())
result = qa_chain({"query": "What are the main topics?"})
print(result["result"])
'''
print('LangChain RAG pipeline:')
print(langchain_example)

In [ ]:
# Evaluation metrics for QA
from collections import Counter

def exact_match(prediction, ground_truth):
    return int(prediction.strip().lower() == ground_truth.strip().lower())

def f1_token_score(prediction, ground_truth):
    pred_tokens = prediction.lower().split()
    truth_tokens = ground_truth.lower().split()
    common = Counter(pred_tokens) & Counter(truth_tokens)
    num_common = sum(common.values())
    if num_common == 0:
        return 0.0
    precision = num_common / len(pred_tokens)
    recall = num_common / len(truth_tokens)
    return 2 * precision * recall / (precision + recall)

preds = ['Paris is the capital', 'The sky is blue', 'Python is a language']
truths = ['Paris', 'blue', 'Python programming language']
for p, t in zip(preds, truths):
    print(f'Pred: {p:35s} | Truth: {t:25s} | F1: {f1_token_score(p,t):.3f}')

## Exercises
1. Expand the rule-based chatbot with 10+ more intents.
2. Build a QA system on a PDF document using LangChain + FAISS.
3. Evaluate your QA system using EM and F1 on 20 test questions.


## Summary
- Chatbots range from rule-based to retrieval-augmented generation (RAG).
- TF-IDF retrieval finds relevant documents by keyword similarity.
- LangChain simplifies building LLM-powered applications.
